In [1]:
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'

import cv2
import sys
import torch

import psdr_jit as psdr
import drjit
from drjit.cuda.ad import Float as FloatD, Matrix4f as Matrix4fD
from drjit.cuda import Float as FloatC, Matrix4f as Matrix4fC

In [2]:
sc = psdr.Scene()
sc.opts.spp = 32 # Interior Term
sc.opts.sppe = 32 # Primary Edge
sc.opts.sppse = 32 # Secondary Edge
sc.opts.height = 1024
sc.opts.width = 1024

integrator = psdr.PathTracer(3)	


sensor = psdr.PerspectiveCamera(60, 0.000001, 10000000.)
to_world = Matrix4fD([[1.,0.,0.,208.],
                     [0.,1.,0.,273.],
                     [0.,0.,1.,-800.],
                     [0.,0.,0.,1.],])
sensor.to_world = to_world
sc.add_Sensor(sensor)

sc.add_BSDF(psdr.DiffuseBSDF([0.0, 0.0, 0.0]), "light")
sc.add_BSDF(psdr.DiffuseBSDF(), "cat")
sc.add_BSDF(psdr.DiffuseBSDF([0.95, 0.95, 0.95]), "white")
sc.add_BSDF(psdr.DiffuseBSDF([0.20, 0.90, 0.20]), "green")
sc.add_BSDF(psdr.DiffuseBSDF([0.90, 0.20, 0.20]), "red")

sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_luminaire.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,-0.5],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "light", psdr.AreaLight([20.0, 20.0, 8.0]))
sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_smallbox.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "cat", None)
sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_largebox.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "cat", None)
sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_floor.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "white", None)
sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_ceiling.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "white", None)
sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_back.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "white", None)
sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_greenwall.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "green", None)
sc.add_Mesh("../psdr-jit/tutorials/data/cbox/cbox_redwall.obj", Matrix4fC([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.]]), "red", None)

P = FloatD(0.)
drjit.enable_grad(P)

sc.param_map["Mesh[0]"].set_transform(Matrix4fD([[1.,0.,0.,P*100.],[0.,1.,0.,0.],[0.,0.,1.,0.],[0.,0.,0.,1.],]))


sc.configure()
sc.configure([0])

img = integrator.renderD(sc, 0)
org_img = img.numpy().reshape((sc.opts.width, sc.opts.height, 3))
output = cv2.cvtColor(org_img, cv2.COLOR_RGB2BGR)
cv2.imwrite("psdr_jit_forward.exr", output)


drjit.set_grad(P, 1.0)
drjit.forward_to(img)
diff_img = drjit.grad(img)
diff_img = diff_img.numpy().reshape((sc.opts.width, sc.opts.height, 3))
output = cv2.cvtColor(diff_img, cv2.COLOR_RGB2BGR)
cv2.imwrite("psdr_jit_diff_debug.exr", output)

add_Sensor: PerspectiveCamera
add_BSDF: Diffuse light
add_BSDF: Diffuse cat
add_BSDF: Diffuse white
add_BSDF: Diffuse green
add_BSDF: Diffuse red
add_Mesh: 0
add Area light
add_Mesh: 1
add_Mesh: 2
add_Mesh: 3
add_Mesh: 4
add_Mesh: 5
add_Mesh: 6
add_Mesh: 7
[Scene] resolution: 1024 1024
[Scene] AABB: [lower = [[0, -9.1e-05, -800]], upper = [[556, 548.8, 559.2]]]
[Scene] (1) primary edges initialized.
[Scene] 66 secondary edges initialized.
[Scene] Configured in 0.045856 seconds.
[Scene] resolution: 1024 1024
[Scene] AABB: [lower = [[0, -9.1e-05, -800]], upper = [[556, 548.8, 559.2]]]
[Scene] (42) primary edges initialized.
[Scene] 66 secondary edges initialized.
[Scene] Configured in 0.00533594 seconds.
render_secondary_edges
[PathTracer] Rendered in 0.435207 seconds.


True

In [3]:
# use_face_normal=True: skips averaged vertex normals, uses actual face normal per triangle.
# This is correct for hard-edged shapes like boxes — averaged normals at 90° edges are wrong.
for mesh_idx in [1, 2]:
    sc.param_map[f"Mesh[{mesh_idx}]"].use_face_normal = True

sc.configure()
sc.configure([0])

img = integrator.renderD(sc, 0)
org_img = img.numpy().reshape((sc.opts.width, sc.opts.height, 3))
output = cv2.cvtColor(org_img, cv2.COLOR_RGB2BGR)
cv2.imwrite("psdr_jit_forward.exr", output)

[Scene] resolution: 1024 1024
[Scene] AABB: [lower = [[0, -9.1e-05, -800]], upper = [[556, 548.8, 559.2]]]
[Scene] (1) primary edges initialized.
[Scene] 66 secondary edges initialized.
[Scene] Configured in 0.00695876 seconds.
[Scene] resolution: 1024 1024
[Scene] AABB: [lower = [[0, -9.1e-05, -800]], upper = [[556, 548.8, 559.2]]]
[Scene] (45) primary edges initialized.
[Scene] 66 secondary edges initialized.
[Scene] Configured in 0.00523185 seconds.
render_secondary_edges
[PathTracer] Rendered in 0.399568 seconds.


True

## Shape Optimization with IRTK — mirroring Neural PBIR `run_shape_IT3.py`

This replicates stage 3 of Neural PBIR on the Cornell box using the exact same IRTK classes:

| Neural PBIR class | Role here |
|---|---|
| `irtk.scene.Scene` + `Mesh/DiffuseBRDF/HDRFilm/…` | Named component dict — mirrors `NeuralPBIRDataset.get_scene()` |
| `irtk.renderer.Renderer` | `torch.autograd.Function` wrapping `PSDRJITConnector.renderC/renderD` |
| `models.shape_ls.ShapeLS` | LargeSteps optimizer on `mesh['v']`; `set_data()` marks vertices as updated for the connector |
| `opt.py` loop inline | `zero_grad → set_data → scene.configure → render_opt → loss → backward → step` |

**Gradient flow per iteration:**  
`loss.backward()` → `RenderFunction.backward` receives `∂loss/∂image` → `PSDRJITConnector.renderD` re-renders with `drjit.enable_grad(vertex_positions)`, `drjit.backward`, `.torch()` bridges grad to `mesh.v.grad` → `LargeStepsOptimizer.step()` chains `V.grad → u.grad` via `from_differential`, AdamUniform on `u`

In [ ]:
import sys
import torch
import numpy as np
import gin
gin.enter_interactive_mode()

NPBIR_PBR = '/home/ahc/Documents/metrology_ir/DigitalTwinCatalog/neural_pbir/pbir'
if NPBIR_PBR not in sys.path:
    sys.path.insert(0, NPBIR_PBR)

from irtk.scene import Scene, Mesh, DiffuseBRDF, HDRFilm, Integrator, PerspectiveCamera
from irtk.renderer import Renderer
from irtk.io import write_image, to_torch_f
from irtk.loss import l1_loss
from models.shape_ls import ShapeLS

CBOX = '/home/ahc/Documents/psdr-jit/tutorials/data/cbox'

def load_obj_tri(path):
    """Load an OBJ (triangles or quads) → (v, f, uv, fuv) numpy arrays.
    gpytoolbox.read_mesh rejects quads, so we parse and fan-triangulate here."""
    verts, uvs, faces, faces_uv = [], [], [], []
    with open(path) as fh:
        for line in fh:
            t = line.split()
            if not t: continue
            if t[0] == 'v':
                verts.append([float(x) for x in t[1:4]])
            elif t[0] == 'vt':
                uvs.append([float(x) for x in t[1:3]])
            elif t[0] == 'f':
                vi, ti = [], []
                for tok in t[1:]:
                    p = tok.split('/')
                    vi.append(int(p[0]) - 1)
                    ti.append(int(p[1]) - 1 if len(p) > 1 and p[1] else 0)
                for i in range(1, len(vi) - 1):
                    faces.append([vi[0], vi[i], vi[i+1]])
                    faces_uv.append([ti[0], ti[i], ti[i+1]])
    v   = np.array(verts,    dtype=np.float32)
    f   = np.array(faces,    dtype=np.int32)
    uv  = np.array(uvs,      dtype=np.float32) if uvs else np.zeros((len(verts), 2), np.float32)
    fuv = np.array(faces_uv, dtype=np.int32)   if uvs else f.copy()
    return v, f, uv, fuv

def mesh_from_obj(path, **kwargs):
    v, f, uv, fuv = load_obj_tri(path)
    # can_change_topology=True forces the connector to use load_raw(), which is the
    # only code path that actually applies use_face_normal to the psdr_jit mesh.
    # The False path (write temp .obj → add_Mesh) silently ignores use_face_normal.
    kwargs.setdefault('can_change_topology', True)
    return Mesh(v, f, uv, fuv, **kwargs)

# ---- Build IRTK Scene ----
# Mirrors NeuralPBIRDataset.get_scene() in neural_pbir_dataset.py.
# Component order matters: BSDFs must be set before the meshes that reference them.
irtk_scene = Scene()

irtk_scene.set('film',       HDRFilm(width=512, height=512))
irtk_scene.set('integrator', Integrator('path', {'max_depth': 3, 'hide_emitters': False}))
irtk_scene.set('sensor 0',   PerspectiveCamera(
    fov=60,
    to_world=torch.tensor([[1.,0.,0.,208.],[0.,1.,0.,273.],[0.,0.,1.,-800.],[0.,0.,0.,1.]]),
    near=1e-6, far=1e7
))

irtk_scene.set('mat_cat',   DiffuseBRDF([0.5,  0.5,  0.5 ]))
irtk_scene.set('mat_white', DiffuseBRDF([0.95, 0.95, 0.95]))
irtk_scene.set('mat_green', DiffuseBRDF([0.20, 0.90, 0.20]))
irtk_scene.set('mat_red',   DiffuseBRDF([0.90, 0.20, 0.20]))

luminaire_xfm = torch.tensor([[1.,0.,0.,0.],[0.,1.,0.,-0.5],[0.,0.,1.,0.],[0.,0.,0.,1.]])
irtk_scene.set('luminaire', mesh_from_obj(f'{CBOX}/cbox_luminaire.obj',
    radiance=[20., 20., 8.], to_world=luminaire_xfm, use_face_normal=True))

irtk_scene.set('mesh',     mesh_from_obj(f'{CBOX}/cbox_smallbox.obj', mat_id='mat_cat',   use_face_normal=True))
irtk_scene.set('largebox', mesh_from_obj(f'{CBOX}/cbox_largebox.obj', mat_id='mat_cat',   use_face_normal=True))
irtk_scene.set('floor',    mesh_from_obj(f'{CBOX}/cbox_floor.obj',    mat_id='mat_white', use_face_normal=True))
irtk_scene.set('ceiling',  mesh_from_obj(f'{CBOX}/cbox_ceiling.obj',  mat_id='mat_white', use_face_normal=True))
irtk_scene.set('back',     mesh_from_obj(f'{CBOX}/cbox_back.obj',     mat_id='mat_white', use_face_normal=True))
irtk_scene.set('green',    mesh_from_obj(f'{CBOX}/cbox_greenwall.obj', mat_id='mat_green', use_face_normal=True))
irtk_scene.set('red',      mesh_from_obj(f'{CBOX}/cbox_redwall.obj',   mat_id='mat_red',   use_face_normal=True))

print("IRTK scene components:", list(irtk_scene.components.keys()))

render_opt = Renderer('psdr_jit', render_options={
    'spp': 16, 'sppe': 8, 'sppse': 4, 'npass': 1, 'log_level': 0
})
render_vis = Renderer('psdr_jit', render_options={
    'spp': 64, 'sppe': 0, 'sppse': 0, 'npass': 1, 'log_level': 0
})
print("Renderers ready")

In [5]:
# render_vis(scene, sensor_ids=[0]) → (1, H, W, 3)
# First call: PSDRJITConnector.update_scene_objects builds the psdr_jit scene and
# calls configure() (full, once). Subsequent calls only call configure([sensor_ids]).
target_imgs = render_vis(irtk_scene, sensor_ids=[0])
target = target_imgs[0]  # (512, 512, 3) on cuda
write_image('shape_target_irtk.exr', target)
print(f"Target rendered: {tuple(target.shape)}")

V_gt = irtk_scene['mesh.v'].detach().clone()
print(f"Small box verts: {V_gt.shape}  Y=[{V_gt[:,1].min():.0f}, {V_gt[:,1].max():.0f}]")

# Perturb small box +70 in Y.
# ParamGroup.__setitem__ automatically calls mark_updated('v'), so the connector
# will push the new V on the next render (just like ShapeLS.set_data() does).
irtk_scene['mesh']['v'] = irtk_scene['mesh']['v'].detach() + to_torch_f([[0., 70., 0.]])

# Render perturbed initial state
init_imgs = render_vis(irtk_scene, sensor_ids=[0])
write_image('shape_init_irtk.exr', init_imgs[0])
print("Rendered perturbed initial state → shape_init_irtk.exr")

Target rendered: (512, 512, 3)
Small box verts: torch.Size([8, 3])  Y=[0, 165]
Rendered perturbed initial state → shape_init_irtk.exr


In [6]:
from pathlib import Path
from tqdm import tqdm

result_path = Path('shape_opt_irtk')
result_path.mkdir(exist_ok=True, parents=True)

# Bump sppe/sppse to match Neural PBIR stage 3 gin config (sppe=16, sppse=8)
# and scale up lr — the Cornell box scene units are in hundreds (box Y up to 165)
# so lr=5e-4 from NeuralPBIR (designed for unit-scale meshes) is far too small here.
# lmbda=1 reduces Laplacian stiffness; with only 8 verts it was 20× over-regularizing.
render_opt.render_options.update({'spp': 16, 'sppe': 16, 'sppse': 8})

model = ShapeLS(irtk_scene, mesh_id='mesh', optimizer_kwargs={'lr': 5.0, 'lmbda': 1})

# ---- Optimization loop: directly mirrors opt.py's optimize() ----
num_epochs      = 20
max_iter        = 100
checkpoint_iter = 25

opt_sensor_ids = torch.arange(1, dtype=torch.int64)
loss_record = []
iter_count  = 0
pbar = tqdm(total=max_iter)

for _ in range(1, num_epochs + 1):
    sensor_perm = opt_sensor_ids[torch.randperm(len(opt_sensor_ids))]

    for sensor_id in sensor_perm:
        model.zero_grad()
        model.set_data()
        irtk_scene.configure()

        opt_image = render_opt(irtk_scene, sensor_ids=[int(sensor_id)], integrator_id=0)[0]

        image_loss = l1_loss(target, opt_image)
        reg_loss   = model.get_regularization()
        loss       = image_loss + reg_loss

        loss.backward()

        if iter_count < 3 or iter_count % checkpoint_iter == 0:
            v = irtk_scene['mesh.v']
            grad = v.grad
            print(f"  [{iter_count:3d}] L1={image_loss.item():.5f}  "
                  f"grad_norm={grad.norm():.3e}  mean_Y={v.detach()[:,1].mean():.1f}  (GT={V_gt[:,1].mean():.1f})")

        model.step()

        loss_record.append(loss.item())
        iter_count += 1
        pbar.update(1)
        pbar.set_postfix({'L1': f'{image_loss.item():.5f}'})

        if iter_count == 1 or iter_count % checkpoint_iter == 0:
            model.write_results(result_path / str(iter_count))
            with torch.no_grad():
                vis = render_vis(irtk_scene, sensor_ids=[0])
            write_image(result_path / str(iter_count) / 'vis.exr', vis[0])

        if iter_count >= max_iter:
            break
    if iter_count >= max_iter:
        break

pbar.close()
torch.save(loss_record, result_path / 'loss.pt')

V_final = irtk_scene['mesh.v'].detach()
print(f"\nDone. Final L1={loss_record[-1]:.5f}")
print(f"V mean_Y: {V_final[:,1].mean():.1f}  (GT: {V_gt[:,1].mean():.1f})")

  0%|          | 0/100 [00:00<?, ?it/s]

  1%|          | 1/100 [00:03<05:42,  3.46s/it]

  1%|          | 1/100 [00:03<05:42,  3.46s/it, L1=0.00962]

  [  0] L1=0.00962  grad_norm=4.890e-05  mean_Y=152.5  (GT=82.5)


  2%|▏         | 2/100 [00:09<07:47,  4.77s/it, L1=0.00962]

...

 19%|█▉        | 19/100 [01:34<06:45,  5.01s/it, L1=0.00619]

 20%|██        | 20/100 [01:39<06:40,  5.01s/it, L1=0.00619]

 20%|██        | 20/100 [01:39<06:40,  5.01s/it, L1=0.00631]

 20%|██        | 20/100 [01:39<06:36,  4.96s/it, L1=0.00631]


Done. Final L1=0.00631
V mean_Y: 76.5  (GT: 82.5)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ── Reset mesh to the +70Y perturbed state ─────────────────────────────────────
# ParamGroup.__setitem__ auto-calls mark_updated('v'), so the connector will push
# the new V on the next render without needing an explicit mark_updated call.
irtk_scene['mesh']['v'] = V_gt + to_torch_f([[0., 70., 0.]])

model_anim = ShapeLS(irtk_scene, mesh_id='mesh', optimizer_kwargs={'lr': 5.0, 'lmbda': 1})
render_opt.render_options.update({'spp': 16, 'sppe': 16, 'sppse': 8})

# ── Optimization loop — capture a vis render every `save_every` iters ──────────
num_iters_anim = 60
save_every     = 2

frames     = []   # list of (H,W,3) numpy arrays
mean_ys    = []   # for the subtitle

print("Collecting frames…")
for it in range(num_iters_anim):
    model_anim.zero_grad()
    model_anim.set_data()
    irtk_scene.configure()

    opt_image  = render_opt(irtk_scene, sensor_ids=[0], integrator_id=0)[0]
    image_loss = l1_loss(target, opt_image)
    image_loss.backward()
    model_anim.step()

    if it % save_every == 0:
        with torch.no_grad():
            frame = render_vis(irtk_scene, sensor_ids=[0])[0]
        frames.append(frame.cpu().numpy().clip(0, 1))
        mean_ys.append(irtk_scene['mesh.v'].detach()[:, 1].mean().item())
        if it % 10 == 0:
            print(f"  iter {it:3d}  L1={image_loss.item():.5f}  mean_Y={mean_ys[-1]:.1f}")

print(f"Collected {len(frames)} frames.")

# ── Build matplotlib animation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.patch.set_facecolor('black')

ax_tgt, ax_cur = axes
for ax in axes:
    ax.axis('off')
    ax.set_facecolor('black')

im_tgt = ax_tgt.imshow(target.cpu().numpy().clip(0, 1), vmin=0, vmax=1)
ax_tgt.set_title('Target', color='white', fontsize=13)

im_cur = ax_cur.imshow(frames[0], vmin=0, vmax=1)
ttl = ax_cur.set_title(f'Iter 0  mean_Y={mean_ys[0]:.1f}  GT={V_gt[:,1].mean():.1f}',
                        color='white', fontsize=11)

def update(i):
    im_cur.set_data(frames[i])
    ttl.set_text(f'Iter {i*save_every:3d}  mean_Y={mean_ys[i]:.1f}  GT={V_gt[:,1].mean():.1f}')
    return im_cur, ttl

anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=150, blit=True)
plt.tight_layout()

anim.save('shape_opt_animation.gif', writer='pillow', fps=6, dpi=100)
plt.close()
print("Saved → shape_opt_animation.gif")

# ── Display inline ──────────────────────────────────────────────────────────────
HTML(anim.to_jshtml())